# PST-Parser su Colab: run esplicite e recuperabili

Il notebook invoca la CLI del progetto. Carica un archivio ZIP della sorgente esatta da eseguire, comprendente `src`, `configs`, `prompts`, `data`, `pyproject.toml`, `uv.lock`, `README.md` e `LICENSE`. **Escludi `.env`, credenziali, `.git`, `.venv`, cache e vecchi output.** La revisione locale e il lock vengono conservati tramite digest; non si seleziona automaticamente una revisione remota mobile.

Seleziona un runtime GPU compatibile con il modello. Memoria e tempi dipendono dal backend e dai dati: verifica prima con il pilot. Google Drive conserva i pesi in cache e tutti gli artefatti della RUN_ID fin dal primo stadio. L'accesso al modello dipende dalla sua licenza; eventuali credenziali vanno nei secret del runtime, mai nell'archivio.


In [ ]:
import subprocess
subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], check=True)


In [ ]:
from pathlib import Path
from google.colab import files
import hashlib
import os
import zipfile

uploaded = files.upload()  # selezionare un solo archivio sorgente ZIP
if len(uploaded) != 1:
    raise ValueError("Selezionare un solo archivio ZIP della sorgente")
archive_name, archive_bytes = next(iter(uploaded.items()))
SOURCE_SHA256 = hashlib.sha256(archive_bytes).hexdigest()
SOURCE_ROOT = Path("/content/pst-parser-source")
if SOURCE_ROOT.exists():
    raise FileExistsError("Sorgente gia estratta: riutilizzare il runtime solo dopo verifica esplicita")
SOURCE_ROOT.mkdir()
with zipfile.ZipFile(archive_name) as bundle:
    for member in bundle.infolist():
        target = (SOURCE_ROOT / member.filename).resolve()
        if not target.is_relative_to(SOURCE_ROOT.resolve()):
            raise ValueError("Archivio con percorso esterno alla directory sorgente")
        if Path(member.filename).name == ".env":
            raise ValueError("Rimuovere .env dall'archivio sorgente")
    bundle.extractall(SOURCE_ROOT)
roots = list(SOURCE_ROOT.rglob("pyproject.toml"))
if len(roots) != 1:
    raise ValueError("L'archivio deve contenere un solo progetto Python")
PROJECT = roots[0].parent
os.chdir(PROJECT)
print("source sha256:", SOURCE_SHA256)


In [ ]:
import sys
subprocess.run([sys.executable, "-m", "pip", "install", "uv==0.11.15"], check=True)
subprocess.run(["uv", "sync", "--frozen", "--python", "3.11", "--extra", "cu128", "--extra", "train", "--extra", "unsloth", "--extra", "structured"], check=True)
UV_RUN = ["uv", "run", "--frozen", "--extra", "cu128", "--extra", "train", "--extra", "unsloth", "--extra", "structured"]
subprocess.run([*UV_RUN, "pstparser", "--version"], check=True)


In [ ]:
from google.colab import drive
import json
import re

drive.mount("/content/drive")
os.environ["HF_HOME"] = "/content/drive/MyDrive/hf-cache"
RUN_ID = "parser-pilot-001"  # cambiare per ogni NUOVA run
MODEL_REVISION = "INSERIRE_COMMIT_MODELLO_40_CARATTERI"
if not re.fullmatch(r"[A-Za-z0-9_-]+", RUN_ID):
    raise ValueError("RUN_ID deve contenere solo lettere, cifre, _ e -")
if not re.fullmatch(r"[0-9a-fA-F]{40}", MODEL_REVISION):
    raise ValueError("Fissare una revisione concreta del modello prima di eseguire")
RUN_ROOT = Path("/content/drive/MyDrive/pst-parser-runs") / RUN_ID
RUN_ROOT.mkdir(parents=True, exist_ok=True)
CONFIG = RUN_ROOT / "run-config.yaml"
IDENTITY = RUN_ROOT / "run-identity.json"
identity = {"run_id": RUN_ID, "source_sha256": SOURCE_SHA256, "model_revision": MODEL_REVISION}
if IDENTITY.exists():
    if json.loads(IDENTITY.read_text()) != identity:
        raise ValueError("RUN_ID gia associata a sorgente/modello diversi")
else:
    IDENTITY.write_text(json.dumps(identity, indent=2))


## Pipeline

Ogni comando usa la stessa configurazione risolta e termina la cella con un errore se lo stadio fallisce. I percorsi sono associati a RUN_ID; nessun adapter viene selezionato da una ricerca dell'ultima directory.

Per recuperare una sessione interrotta, ricarica lo stesso archivio e imposta la stessa RUN_ID e revisione modello. Verifica i manifest degli stadi gia completati e salta le relative celle. Una directory parziale senza manifest non e uno stadio completato: conservarla per diagnosi e utilizzare una nuova RUN_ID per una nuova esecuzione. Non viene promessa una ripresa automatica del training dal checkpoint.

Un microbatch diverso con accumulo compensato mantiene il batch effettivo, non garantisce identica traiettoria numerica. Eventuali override vanno stabiliti prima del freeze della configurazione.


In [ ]:
overrides = [
    f"name={json.dumps(RUN_ID)}",
    f"model.revision={json.dumps(MODEL_REVISION)}",
    f"data.processed_dir={(RUN_ROOT / 'prepared').as_posix()}",
    f"data.split.output_dir={(RUN_ROOT / 'splits').as_posix()}",
    f"training.output_dir={(RUN_ROOT / 'training').as_posix()}",
]
# Il progetto risolve e valida la configurazione nel proprio ambiente bloccato.
resolve = "import json,sys,yaml; from pstparser.config import load_experiment; c=load_experiment(sys.argv[1],overrides=json.loads(sys.argv[2])); print(yaml.safe_dump(c.model_dump(mode='json'),sort_keys=False))"
resolved = subprocess.run([*UV_RUN, "python", "-c", resolve, "configs/experiments/baseline.yaml", json.dumps(overrides)], check=True, capture_output=True, text=True).stdout
if CONFIG.exists():
    if CONFIG.read_text() != resolved:
        raise ValueError("Configurazione congelata diversa: utilizzare una nuova RUN_ID")
else:
    CONFIG.write_text(resolved)

def run(*arguments):
    subprocess.run([*UV_RUN, "pstparser", *map(str, arguments), "--config", str(CONFIG)], check=True)

if (RUN_ROOT / "prepared" / "prepare_manifest.json").is_file():
    print("Preparazione gia completata per questa identita/configurazione; file preservati")
else:
    run("prepare-data")


In [ ]:
# Pilot sul validation set: non consultare il test per mettere a punto la pipeline.
run("generate", "--run-dir", RUN_ROOT / "smoke", "--split", "val", "--limit", 2)


In [ ]:
TRAIN_DIR = RUN_ROOT / "training"
run("train", "--run-dir", TRAIN_DIR)


In [ ]:
TRAIN_DIR = RUN_ROOT / "training"
ADAPTER = TRAIN_DIR / "adapter"
if not (TRAIN_DIR / "train_manifest.json").is_file() or not ADAPTER.is_dir():
    raise RuntimeError("Training incompleto: manca manifest o adapter della RUN_ID selezionata")
TUNED_DIR = RUN_ROOT / "tuned"
run("generate", "--adapter", ADAPTER, "--run-dir", TUNED_DIR)
run("score", "--predictions", TUNED_DIR / "predictions.jsonl", "--run-dir", TUNED_DIR / "score")
run("align", "--predictions", TUNED_DIR / "predictions.jsonl", "--run-dir", TUNED_DIR / "align")


## Esecuzioni di confronto

La baseline senza adapter e il decoding vincolato sono condizioni distinte sullo stesso test congelato. Avviarle solo quando pilot e protocollo sono fissati. Il confronto base/tuned misura il contributo del fine-tuning del parser, non quello dei prompt gerarchici sulla generazione del codice.


In [ ]:
BASE_DIR = RUN_ROOT / "base"
run("generate", "--run-dir", BASE_DIR)
run("score", "--predictions", BASE_DIR / "predictions.jsonl", "--run-dir", BASE_DIR / "score")
run("align", "--predictions", BASE_DIR / "predictions.jsonl", "--run-dir", BASE_DIR / "align")


In [ ]:
# Il vincolo controlla il formato durante la decodifica, ma un output troncato
# resta un fallimento: la validita va misurata anche in questa condizione.
CONSTRAINED_DIR = RUN_ROOT / "constrained"
run("generate", "--adapter", ADAPTER, "--run-dir", CONSTRAINED_DIR, "--set", "inference.structured_output=true")
run("score", "--predictions", CONSTRAINED_DIR / "predictions.jsonl", "--run-dir", CONSTRAINED_DIR / "score")
run("align", "--predictions", CONSTRAINED_DIR / "predictions.jsonl", "--run-dir", CONSTRAINED_DIR / "align")


## Recupero degli artefatti

Gli artefatti sono gia su Drive sotto RUN_ID, anche se una cella successiva fallisce. Questa cella crea inoltre un archivio di trasporto con adapter finale, dataset preparato, split e relativi hash, configurazione risolta, lock, sorgenti Python, prompt e manifest di tutti gli stadi. I checkpoint intermedi sono esclusi; conservarli separatamente se servono per una futura ripresa del training. Nessuna credenziale e inclusa.


In [ ]:
import zipfile

ARCHIVE = RUN_ROOT / f"{RUN_ID}-artifacts.zip"
keep_names = {"predictions.jsonl", "results.json", "eval_details.jsonl", "alignments.jsonl", "training_metrics.jsonl", "run-config.yaml", "run-identity.json", "records.jsonl", "quality_report.json", "train.json", "val.json", "test.json", "generations.jsonl"}
with zipfile.ZipFile(ARCHIVE, "x", compression=zipfile.ZIP_DEFLATED) as bundle:
    for path in sorted(RUN_ROOT.rglob("*")):
        if not path.is_file() or path == ARCHIVE:
            continue
        relative = path.relative_to(RUN_ROOT)
        if any(part.startswith("checkpoint-") for part in relative.parts):
            continue
        if path.name in keep_names or path.suffix == ".jsonl" or path.name.endswith("_manifest.json") or "adapter" in relative.parts:
            bundle.write(path, f"run/{relative.as_posix()}")
    for directory, pattern in (("src", "*.py"), ("configs", "*.yaml"), ("prompts", "*.md")):
        for path in sorted((PROJECT / directory).rglob(pattern)):
            bundle.write(path, f"source/{path.relative_to(PROJECT).as_posix()}")
    for name in ("uv.lock", "pyproject.toml", "README.md", "LICENSE"):
        bundle.write(PROJECT / name, f"source/{name}")
print("Artefatti conservati:", RUN_ROOT)
print("Archivio:", ARCHIVE)
